<a href="https://colab.research.google.com/github/siddhartha-sai-17/Gen-AI/blob/main/assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 1: Document Loading

First, let's install the necessary libraries: `pypdf` for reading PDF files and `reportlab` for generating sample PDF files.

In [ ]:
%%capture
!pip install pypdf reportlab

Now, let's create the sample PDF documents based on the provided dataset names.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
import os

def create_sample_pdf(filename, content, num_pages=1):
    c = canvas.Canvas(filename, pagesize=letter)
    for i in range(num_pages):
        c.drawString(100, 750 - (i % 20) * 20, f"Page {i+1}: {content}")
        if i < num_pages - 1:
            c.showPage()
    c.save()


pdf_documents = [
    ("Employee Handbook.pdf", "This is the Employee Handbook. It contains general information about company policies and employee conduct. ", 5),
    ("Leave Policy.pdf", "This document outlines the company's leave policy, including sick leave, annual leave, and parental leave.", 3),
    ("Travel Policy.pdf", "This policy covers guidelines for business travel, expenses, and reimbursement procedures.", 2),
    ("Work From Home Policy.pdf", "Details regarding the company's work-from-home policy, eligibility, and expectations.", 4),
    ("Medical Insurance Policy.pdf", "Information about employee medical insurance, coverage details, and claim procedures.", 6)
]

for filename, content, num_pages in pdf_documents:
    create_sample_pdf(filename, content * 50, num_pages) # Repeat content to make it longer
    print(f"Created {filename} with {num_pages} pages.")

Created Employee Handbook.pdf with 5 pages.
Created Leave Policy.pdf with 3 pages.
Created Travel Policy.pdf with 2 pages.
Created Work From Home Policy.pdf with 4 pages.
Created Medical Insurance Policy.pdf with 6 pages.


Next, we will load these PDF files, extract their text content, and display the required statistics: File Name, Number of Pages, Total Characters, and Total Words.

In [ ]:
from pypdf import PdfReader
import pandas as pd

def load_and_analyze_pdfs(pdf_files):
    data = []
    for pdf_file in pdf_files:
        try:
            reader = PdfReader(pdf_file)
            num_pages = len(reader.pages)
            text_content = ""
            for page in reader.pages:
                text_content += page.extract_text() if page.extract_text() else ""
            total_characters = len(text_content)
            total_words = len(text_content.split())

            data.append({
                "File Name": pdf_file,
                "Number of Pages": num_pages,
                "Total Characters": total_characters,
                "Total Words": total_words
            })
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    return pd.DataFrame(data)

# Get the list of created PDF files
created_pdf_files = [filename for filename, _, _ in pdf_documents]

# Load and analyze the PDFs
df_document_stats = load_and_analyze_pdfs(created_pdf_files)

# Display the document statistics
display(df_document_stats)

,File Name,Number of Pages,Total Characters,Total Words
0,Employee Handbook.pdf,5,27045,3760
1,Leave Policy.pdf,3,15927,2109
2,Travel Policy.pdf,2,9018,1006
3,Work From Home Policy.pdf,4,17036,1612
4,Medical Insurance Policy.pdf,6,25554,2718


## Task 2: Document Chunking

In this task, we split the extracted policy text into manageable segments (chunks). We use a **Chunk Size of 500 characters** and a **Chunk Overlap of 100 characters** to ensure context is preserved between segments.

In [ ]:
def fixed_size_chunking(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_content = text[start:end]
        chunks.append({
            "Content": chunk_content,
            "Length": len(chunk_content)
        })
        if end >= len(text): break
        start += (chunk_size - overlap)
    return chunks

# Extract text from one of the documents for demonstration
reader = PdfReader("Employee Handbook.pdf")
full_text = "".join([p.extract_text() for p in reader.pages if p.extract_text()])

# Apply Fixed Size Chunking
fixed_chunks = fixed_size_chunking(full_text)

print(f"--- Fixed Size Chunking (First 3 Chunks) ---")
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"Chunk {i+1}:")
    print(f"Content: {chunk['Content'][:100]}...")
    print(f"Length: {chunk['Length']}\n")

--- Fixed Size Chunking (First 3 Chunks) ---
Chunk 1:
Content: Page 1: This is the Employee Handbook. It contains general information about company policies and em...
Length: 500

Chunk 2:
Content:  company policies and employee conduct. This is the Employee Handbook. It contains general informati...
Length: 500

Chunk 3:
Content: ntains general information about company policies and employee conduct. This is the Employee Handboo...
Length: 500



In [ ]:
%%capture
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the Recursive Character Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

recursive_docs = text_splitter.create_documents([full_text])

print(f"--- Recursive Chunking (First 3 Chunks) ---")
for i, doc in enumerate(recursive_docs[:3]):
    print(f"Chunk {i+1}:")
    print(f"Content: {doc.page_content[:100]}...")
    print(f"Length: {len(doc.page_content)}\n")

--- Recursive Chunking (First 3 Chunks) ---
Chunk 1:
Content: Page 1: This is the Employee Handbook. It contains general information about company policies and em...
Length: 490

Chunk 2:
Content: about company policies and employee conduct. This is the Employee Handbook. It contains general info...
Length: 497

Chunk 3:
Content: It contains general information about company policies and employee conduct. This is the Employee Ha...
Length: 499



### Analysis: Why is Chunking Necessary in RAG?

1. **LLM Context Limits**: Large Language Models have a finite 'context window' (the maximum number of tokens they can process at once). Chunking ensures we only send relevant snippets rather than entire libraries.
2. **Retrieval Granularity**: By indexing smaller chunks, the vector search can find the exact sentence or paragraph that answers a query, rather than returning a 50-page document where the answer is hidden.
3. **Noise Reduction**: Smaller, focused chunks prevent the model from getting 'confused' by irrelevant information found in other parts of a large document.
4. **Cost and Speed**: Processing smaller chunks reduces API costs and improves response latency.

## Task 3: Embedding Generation

In this task, we will transform our text chunks into numerical vectors (embeddings) using the `all-MiniLM-L6-v2` model. This allows the computer to perform mathematical operations to find similar pieces of text.

In [ ]:
%%capture
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Extract text from ALL documents
all_texts = []
for pdf_file in created_pdf_files:
    reader = PdfReader(pdf_file)
    text = "".join([p.extract_text() for p in reader.pages if p.extract_text()])
    all_texts.append(text)

# 2. Re-run Recursive Chunking on all documents
all_recursive_docs = text_splitter.create_documents(all_texts)
chunk_texts = [doc.page_content for doc in all_recursive_docs]

# 3. Generate embeddings for all chunks
embeddings = model.encode(chunk_texts)

print(f"--- Embedding Generation Summary ---")
print(f"Total Chunks from all documents: {len(chunk_texts)}")
print(f"Embedding Shape: {embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- Embedding Generation Summary ---
Total Chunks from all documents: 244
Embedding Shape: (244, 384)


### Analysis: How embeddings capture semantic meaning

1. **High-Dimensional Mapping**: Embeddings map words and phrases into a high-dimensional space (384 dimensions for this model). In this space, words with similar meanings are placed close to each other.
2. **Contextual Understanding**: Models like MiniLM are trained on vast amounts of text to understand that 'vacation' and 'leave' or 'doctor' and 'medical' are related, even if they are different words.
3. **Vector Mathematics**: Because the text is now represented as numbers, we can use 'Cosine Similarity' to calculate the distance between a user's question and our policy chunks. A smaller distance (or higher cosine score) indicates a higher semantic match.
4. **Dense Representation**: Unlike a simple keyword search, embeddings create a 'dense' representation where every dimension captures a subtle feature of the language, allowing the system to handle synonyms and paraphrasing effectively.

## Task 4: Build FAISS Vector Database

In this task, we will initialize a FAISS (Facebook AI Similarity Search) index to store our embeddings. This vector database is optimized for finding the nearest neighbors in a high-dimensional space.

In [ ]:
%%capture
!pip install faiss-cpu

In [ ]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"--- Updated FAISS Vector Database ---")
print(f"Number of Stored Vectors: {index.ntotal}")
print(f"Embedding Dimension: {dimension}")

--- Updated FAISS Vector Database ---
Number of Stored Vectors: 244
Embedding Dimension: 384


## Task 5: Semantic Retrieval

Now we will implement the search functionality. This involves converting a user's question into the same embedding space as our documents and using the FAISS index to find the most relevant chunks.

In [ ]:
def retrieve_relevant_chunks(query, top_k=3):
    # 1. Convert query to embedding
    query_embedding = model.encode([query]).astype('float32')

    # 2. Search FAISS index
    # D: Distances (L2 distance, smaller is better)
    # I: Indices of the chunks
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i in range(top_k):
        idx = indices[0][i]
        results.append({
            "chunk_content": chunk_texts[idx],
            "score": float(distances[0][i])
        })
    return results

# Example Queries
queries = [
    "How many casual leaves are available?",
    "Can employees work from home?",
    "How does travel reimbursement work?",
    "Who is covered under medical insurance?"
]

# Execute Retrieval and Display Deliverables
for q in queries:
    print(f"\n{'='*50}")
    print(f"QUERY: {q}")
    print(f"{'='*50}")

    hits = retrieve_relevant_chunks(q)

    for idx, hit in enumerate(hits):
        print(f"Top {idx+1} Chunk (Score: {hit['score']:.4f}):")
        print(f"{hit['chunk_content'][:200]}...")
        print("-" * 30)


QUERY: How many casual leaves are available?
Top 1 Chunk (Score: 1.8968):
the company's leave policy, including sick leave, annual leave, and parental leave.This document outlines the company's leave policy, including sick leave, annual leave, and parental leave.This docume...
------------------------------
Top 2 Chunk (Score: 1.8968):
the company's leave policy, including sick leave, annual leave, and parental leave.This document outlines the company's leave policy, including sick leave, annual leave, and parental leave.This docume...
------------------------------
Top 3 Chunk (Score: 1.8968):
the company's leave policy, including sick leave, annual leave, and parental leave.This document outlines the company's leave policy, including sick leave, annual leave, and parental leave.This docume...
------------------------------

QUERY: Can employees work from home?
Top 1 Chunk (Score: 1.2193):
the company's work-from-home policy, eligibility, and expectations.Details regarding the compan

### Analysis: Why those chunks were retrieved

1. **Vector Proximity**: The retrieved chunks have the smallest L2 distance (lowest numerical score) to the query vector in the 384-dimensional embedding space.
2. **Semantic Alignment**: Even if the exact wording differs (e.g., "work from home" vs "remote work"), the `all-MiniLM-L6-v2` model understands the underlying concepts, ensuring that the WFH policy chunks are ranked highest for the WFH query.
3. **Keyword Importance**: The model places significant weights on key terms like "medical insurance" or "reimbursement," effectively acting as an intelligent keyword search that understands context.
4. **Contextual Overlap**: Because we used recursive chunking with overlap, the system often retrieves chunks that contain both the specific answer and the necessary surrounding context.

## Task 6: Answer Generation

In this final step, we combine the retrieved chunks with the original user query to form a 'Prompt'. This prompt is then processed to generate a coherent answer. This demonstrates the full **Retrieval-Augmented Generation** workflow.

### Task 6: Integrating a Real LLM (Gemma 2B)

Instead of simulated responses, we will now use a quantized version of the **Gemma 2B** model. We'll use the `transformers` library to load the model and generate answers based on the context retrieved from our FAISS index.

In [ ]:
from transformers import pipeline
import torch

# Using TinyLlama as it is open-access and memory-efficient for Colab
# No Hugging Face login or gated access required
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading model: {model_id}...")
generator = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_real_answer(query):
    # 1. Retrieve the top 3 context chunks
    hits = retrieve_relevant_chunks(query, top_k=3)
    context_text = "\n".join([hit['chunk_content'] for hit in hits])

    # 2. Construct the Prompt using Chat Template format
    prompt = f"<|system|>\nYou are an AI assistant that answers HR questions based only on the provided context. If the answer is not in the context, say you don't know.\nContext:\n{context_text}</s>\n<|user|>\n{query}</s>\n<|assistant|>"

    # 3. Generate Answer
    outputs = generator(
        prompt,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.1,
        top_k=50,
        top_p=0.95
    )

    full_response = outputs[0]['generated_text']
    answer = full_response.split("<|assistant|>")[-1].strip()

    print(f"\n{'='*20} RETRIEVED CONTEXT {'='*20}")
    print(context_text[:500] + "...")
    print(f"\n{'='*20} GENERATED ANSWER {'='*20}")
    print(answer)

# Test with a specific query
generate_real_answer("What is the company's policy on annual leave?")

Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_k', 'temperature', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



==================== RETRIEVED CONTEXT ====================
annual leave, and parental leave.This document outlines the company's leave policy, including sick leave, annual leave, and parental leave.
annual leave, and parental leave.This document outlines the company's leave policy, including sick leave, annual leave, and parental leave.
annual leave, and parental leave.This document outlines the company's leave policy, including sick leave, annual leave, and parental leave....

==================== GENERATED ANSWER ====================
The company's policy on annual leave is as follows:

1. Annual leave is a paid leave that employees can take during the year.
2. The company provides employees with 28 days of annual leave per year.
3. Employees can take up to 14 days of annual leave in any 12-month period.
4. Annual leave is not accrued or earned, but employees can use it at any time.
5. Employees can take annual leave during the first 14 days of the year, and they can take up to 14 d

### Final Demonstration: Querying the RAG Assistant

Now that our full pipeline is integrated with a real LLM (**TinyLlama**), let's run several queries to see how the system performs across different policy documents.

In [ ]:
test_queries = [
    "What are the rules for medical insurance claims?",
    "Is there a policy for travel reimbursement?",
    "Tell me about the sick leave policy."
]

for q in test_queries:
    print(f"\n{'#'*60}")
    print(f"USER QUESTION: {q}")
    generate_real_answer(q)
    print(f"{'#'*60}\n")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



############################################################
USER QUESTION: What are the rules for medical insurance claims?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



==================== RETRIEVED CONTEXT ====================
and claim procedures.Information about employee medical insurance, coverage details, and claim procedures.Information about employee medical insurance, coverage details, and claim procedures.Information about employee medical insurance, coverage details, and claim procedures.
and claim procedures.Information about employee medical insurance, coverage details, and claim procedures.Information about employee medical insurance, coverage details, and claim procedures.Information about employee medic...

==================== GENERATED ANSWER ====================
Sure! Here are the general rules for medical insurance claims:

1. The claim must be submitted within the applicable timeframe.

2. The claim must be for the medical services provided by the employee's healthcare provider.

3. The claim must be for the employee's medical expenses incurred during the time period covered by the insurance policy.

4. The claim must be for the

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



==================== RETRIEVED CONTEXT ====================
for business travel, expenses, and reimbursement procedures.This policy covers guidelines for business travel, expenses, and reimbursement procedures.This policy covers guidelines for business travel, expenses, and reimbursement procedures.This policy covers guidelines for business travel, expenses, and reimbursement procedures.This policy covers guidelines for business travel, expenses, and reimbursement procedures.This policy covers guidelines for business travel, expenses, and reimbursement
f...

==================== GENERATED ANSWER ====================
Yes, there is a policy for travel reimbursement. Here's an example of a policy for travel reimbursement:

Policy:

Travel Reimbursement Policy

This policy covers guidelines for travel reimbursement procedures.

1. Eligibility:

Travel reimbursement is available to employees who are authorized to travel for business purposes.

2. Reimbursement Amount:

Travel reimbursement

## Task 7: Complete Reusable RAG Pipeline

Finally, we will wrap the entire logic into a single reusable class. This allows any user to simply input a question and receive a natural language answer based on the indexed corporate policies.

In [ ]:
class EmployeePolicyAssistant:
    def __init__(self, llm_generator, embedding_model, faiss_index, text_chunks):
        self.generator = llm_generator
        self.model = embedding_model
        self.index = faiss_index
        self.chunk_texts = text_chunks

    def ask(self, question, top_k=3):
        # 1. Semantic Retrieval
        query_vec = self.model.encode([question]).astype('float32')
        distances, indices = self.index.search(query_vec, top_k)

        # 2. Context Preparation
        retrieved_context = "\n".join([self.chunk_texts[idx] for idx in indices[0]])

        # 3. LLM Prompt Construction
        prompt = f"<|system|>\nYou are an HR assistant. Use only the context provided to answer. Context:\n{retrieved_context}</s>\n<|user|>\n{question}</s>\n<|assistant|>"

        # 4. Generation
        outputs = self.generator(prompt, max_new_tokens=150, temperature=0.2, do_sample=True)
        answer = outputs[0]['generated_text'].split("<|assistant|>")[-1].strip()

        return {
            "question": question,
            "answer": answer,
            "context_used": retrieved_context[:300] + "..."
        }

# Initialize the Assistant
assistant = EmployeePolicyAssistant(generator, model, index, chunk_texts)

# Final Demonstration
response = assistant.ask("What happens if I need to travel for work?")

print(f"--- FINAL ASSISTANT OUTPUT ---")
print(f"QUESTION: {response['question']}")
print(f"ANSWER: {response['answer']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- FINAL ASSISTANT OUTPUT ---
QUESTION: What happens if I need to travel for work?
ANSWER: If you need to travel for work, you must follow the guidelines for business travel, expenses, and reimbursement procedures outlined in the policy. The policy covers guidelines for business travel, expenses, and reimbursement procedures. If you need to travel for work, you must submit a travel request to your supervisor or HR representative, and they will review the request and approve or deny it based on the guidelines outlined in the policy. If your travel is approved, you will be reimbursed for your expenses related to the trip.


## Task 8: Performance Evaluation

In this final task, we benchmark the RAG system using three key metrics:
- **Retrieval Accuracy**: Measured via the FAISS L2 distance (lower scores indicate higher relevance).
- **Response Time**: Total time taken from query input to final answer generation.
- **Answer Relevance**: Observing if the `TinyLlama` model successfully extracts specific policy details from the synthetic documents.

In [ ]:
import time

# Define the evaluation questions
eval_questions = [
    "How many sick leaves are available?",
    "Can employees permanently work from home?",
    "When are travel expenses reimbursed?",
    "Are family members covered by medical insurance?"
]

def evaluate_assistant(assistant, questions):
    results = []

    print(f"{'Question':<50} | {'Latency (s)':<12} | {'Retrieval Score':<15}")
    print("-" * 85)

    for query in questions:
        start_time = time.time()

        # Get retrieval score separately for evaluation
        query_vec = assistant.model.encode([query]).astype('float32')
        distances, _ = assistant.index.search(query_vec, k=1)
        best_score = distances[0][0]

        # Get full RAG response
        response = assistant.ask(query)

        end_time = time.time()
        latency = end_time - start_time

        results.append({
            "question": query,
            "latency": latency,
            "score": best_score,
            "answer": response['answer']
        })

        print(f"{query[:48]:<50} | {latency:<12.2f} | {best_score:<15.4f}")

    return results

# Run the evaluation
eval_results = evaluate_assistant(assistant, eval_questions)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question                                           | Latency (s)  | Retrieval Score
-------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


How many sick leaves are available?                | 40.57        | 1.5214         


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Can employees permanently work from home?          | 88.89        | 1.2557         


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


When are travel expenses reimbursed?               | 99.03        | 0.8520         
Are family members covered by medical insurance?   | 48.96        | 1.3586         


### Detailed Answer Inspection

Let's look at the generated answers to assess relevance and accuracy.

In [ ]:
for res in eval_results:
    print(f"\nQUERY: {res['question']}")
    print(f"GENERATED ANSWER: {res['answer']}")
    print("-" * 50)


QUERY: How many sick leaves are available?
GENERATED ANSWER: The document does not provide a specific number of sick leaves available. It only mentions that the company offers sick leave, annual leave, and parental leave.
--------------------------------------------------

QUERY: Can employees permanently work from home?
GENERATED ANSWER: I do not have access to the specific policies and procedures of the company you are referring to. However, in general, it is common for companies to allow employees to work from home temporarily or permanently, depending on the situation and the needs of the organization. In some cases, employees may be required to work from home for a specific period of time, such as during a pandemic or a natural disaster. However, in most cases, employees are allowed to work from home permanently, unless there are specific reasons for requiring them to work in the office.
--------------------------------------------------

QUERY: When are travel expenses reimburse